In [2]:
import pandas as pd
import numpy as np

In [18]:
df_raw = pd.read_excel("../data/raw/Telco_customer_churn.xlsx")

# 02. Data Cleaning & Preprocessing

## Business Objective
Clean and standardize the raw Telco customer dataset to ensure data integrity for downstream statistical analysis, visualization, and machine learning. 

### Key Objectives:
1. Fix attribute creation errors and maintain raw data integrity via deep copying.
2. Standardize column names (snake_case) for consistency in SQL and Python.
3. Identify and handle blank/whitespace strings in numerical features (`Total Charges`).
4. Correct data types (e.g., object to numeric/float).
5. Standardize categorical labels (e.g., unifying 'No internet service' to 'No' where applicable).
6. Export the processed clean dataset to `data/processed/customer_churn_clean.csv`.

In [19]:
# Create a copy for cleaning
df_clean = df_raw.copy()

print(f"Dataset Shape: {df_clean.shape}")
df_clean.head()

Dataset Shape: (7043, 33)


,CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,...,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,...,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,...,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,67,2701,Moved
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,...,Month-to-month,Yes,Electronic check,99.65,820.5,Yes,1,86,5372,Moved
3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,...,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,1,84,5003,Moved
4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,...,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.3,Yes,1,89,5340,Competitor had better devices


## 1. Column Name Standardization
Standardizing column names into lowercase `snake_case` removes spaces, special characters, and uppercase inconsistency, making programmatic access and SQL queries seamless.

In [20]:
# Convert column names to lowercase and replace spaces with underscores
df_clean.columns = (
    df_clean.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
    .str.replace('-', '_')
)

print("Standardized Columns:")
print(df_clean.columns.tolist())

Standardized Columns:
['customerid', 'count', 'country', 'state', 'city', 'zip_code', 'lat_long', 'latitude', 'longitude', 'gender', 'senior_citizen', 'partner', 'dependents', 'tenure_months', 'phone_service', 'multiple_lines', 'internet_service', 'online_security', 'online_backup', 'device_protection', 'tech_support', 'streaming_tv', 'streaming_movies', 'contract', 'paperless_billing', 'payment_method', 'monthly_charges', 'total_charges', 'churn_label', 'churn_value', 'churn_score', 'cltv', 'churn_reason']


## 2. Data Type Correction & Missing Values
In the IBM Telco dataset, `Total Charges` (or `total_charges`) often contains empty space strings (`' '`) for brand-new customers with 0 tenure months. We need to coerce these to `NaN` and fill or handle them properly.

In [21]:
# Check total_charges data type
print(f"Original total_charges dtype: {df_clean['total_charges'].dtype}")

# Coerce non-numeric values (like ' ') to NaN
df_clean['total_charges'] = pd.to_numeric(df_clean['total_charges'], errors='coerce')

# Check how many missing values were generated
missing_total = df_clean['total_charges'].isnull().sum()
print(f"Missing values in total_charges after coercion: {missing_total}")

# Fill missing total_charges with 0 (since tenure is 0 for these customers)
df_clean['total_charges'] = df_clean['total_charges'].fillna(0)

# Verify correction
print(f"Updated total_charges dtype: {df_clean['total_charges'].dtype}")

Original total_charges dtype: object
Missing values in total_charges after coercion: 11
Updated total_charges dtype: float64


## 3. Categorical Label Standardization & Duplicate Removal
Check for duplicate customer records and simplify redundant responses (e.g., converting 'No internet service' or 'No phone service' into 'No' where distinct sub-category detail isn't required).

In [22]:
# 1. Remove exact duplicate rows if any exist
duplicates_count = df_clean.duplicated().sum()
print(f"Duplicate rows found: {duplicates_count}")
if duplicates_count > 0:
    df_clean = df_clean.drop_duplicates()
    print("Duplicates removed.")

# 2. Standardize redundant categorical responses across service columns
replace_cols = ['multiple_lines', 'online_security', 'online_backup', 
                'device_protection', 'tech_support', 'streaming_tv', 'streaming_movies']

for col in replace_cols:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].replace({'No internet service': 'No', 'No phone service': 'No'})

print("\nSample Unique Values Check (Online Security):")
print(df_clean['online_security'].value_counts())

Duplicate rows found: 0

Sample Unique Values Check (Online Security):
online_security
No     5024
Yes    2019
Name: count, dtype: int64


## 4. Export Clean Dataset
Save the cleaned DataFrame into `data/processed/customer_churn_clean.csv` for use in EDA, SQL analysis, and Machine Learning phases.

In [23]:
# Save clean dataset
output_path = "../data/processed/customer_churn_clean.csv"
df_clean.to_csv(output_path, index=False)

print(f"✅ Clean dataset successfully saved to: {output_path}")
print(f"Final Shape: {df_clean.shape}")

✅ Clean dataset successfully saved to: ../data/processed/customer_churn_clean.csv
Final Shape: (7043, 33)
